### Importamos la clase Tello desde la librería

In [ ]:
from djitellopy import Tello
import time

### Para conectar el tello, debemos crear un objeto de la clase Tello. Este objeto nos permite conectarnos al drone y realizar diversas operaciones.

In [ ]:
tello = Tello()

for func in dir(tello):
    if callable(getattr(tello, func)):
        print(func)


#### Conectamos el tello

In [ ]:
tello.connect()

### para el aterrizaje seguro, creamos una funcion que nos permita intentar el aterrizaje la cantidad de veces que queramos

In [ ]:
def safe_land():
    print("LANDING...")

    # detener el movimiento
    for _ in range(5):
        tello.send_rc_control(0,0,0,0)
        time.sleep(0.05)

    time.sleep(0.3)

    # intentar land varias veces
    for _ in range(3):
        try:
            tello.land()
            print("LANDED OK")
            return
        except:
            time.sleep(0.5)

    print("FORCED EMERGENCY")

    ## este codigo detiene todos los motores instantaneamente
    tello.emergency()

### para que el drone se eleve 

In [ ]:
tello.takeoff()

### para cortar todo el código con ctrl+c

In [ ]:
import sys, signal

def handler(sig, frame):
    safe_land()
    tello.streamoff()
    tello.end()
    sys.exit(0)

signal.signal(signal.SIGINT, handler)

# FRAMES

### Para leer la imagen del drone, debemos apagar (asegurarnos que esté apagada) y prenderla. 

In [ ]:
import cv2
tello.streamoff()
tello.streamon()
frame_read = tello.get_frame_read()

#### Luego en un while true, debemos adquirir los frames todo el rato !! 

In [ ]:
while True:
    frame = frame_read.frame

# Pero opencv nos lee la imagen en bgr, no en rgb  (estamos dentro del while)

In [ ]:
while True:
    frame = frame_read.frame
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    

# con esto, ya podemos mostrar nuestra imagen y hacer el procesado

In [ ]:
while True:
    frame = frame_read.frame
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    if frame is None:
        continue
    cv2.imshow("Tello", frame)

## lo demas es del manual control 

In [ ]:
while True:
    frame = frame_read.frame
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    if frame is None:
        continue
    cv2.imshow("Tello", frame)


    # =======================
    # LAND
    # =======================


    key = cv2.waitKey(1) & 0xFF
    pressed = set()

    if key != 255:
        pressed.add(chr(key))

    if 'l' in pressed or 'esc' in pressed:
        safe_land()
        break

    
    lr, fb, ud, yaw = 0, 0, 0, 0
    
    speed = 40
    if 'w' in pressed: fb = speed
    if 's' in pressed: fb = -speed
    if 'a' in pressed: lr = -speed
    if 'd' in pressed: lr = speed
    if 'r' in pressed: ud = speed
    if 'f' in pressed: ud = -speed
    if 'q' in pressed: yaw = -speed
    if 'e' in pressed: yaw = speed

    # =======================
    # SEND RC
    # =======================
    tello.send_rc_control(lr, fb, ud, yaw)
